In [0]:
%sql

CREATE OR REPLACE VIEW metric_view_demo.my_schema.order_metrics_sql_2
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: samples.tpch.orders

  joins:
    - name: lineitem
      source: samples.tpch.lineitem
      on: source.o_orderkey = lineitem.l_orderkey
    - name: customer
      source: samples.tpch.customer
      on: source.o_custkey = customer.c_custkey
  dimensions:
    - name: order_date
      expr: o_orderdate
      synonyms:
        - purchase date
        - transaction date
    - name: order_year
      expr: YEAR(o_orderdate)
    - name: customer_name
      expr: customer.c_name
      display_name: Customer Name

  measures:
    - name: total_revenue
      display_name: Total Revenue (USD)
      expr: SUM(o_totalprice)
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
        abbreviation: compact
        hide_group_separator: false
      synonyms:
        - revenue
        - sales
        - gross revenue
    - name: order_count
      display_name: Order Count
      expr: COUNT(*)
    - name: average_revenue
      display_name: Average Revenue (USD)
      expr: measure(total_revenue) / measure(order_count)
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
        abbreviation: compact
        hide_group_separator: false
    - name: total_quantity
      display_name: Total Quantity
      expr: SUM(lineitem.l_quantity)
      format:
        type: number
        decimal_places:
          type: exact
          places: 2
        abbreviation: compact
        hide_group_separator: false
$$

In [0]:
%sql
select
customer_name,
order_year,
measure(total_revenue),
measure(order_count),
measure(average_revenue),
measure(total_quantity)
from
metric_view_demo.my_schema.order_metrics_sql_2
group by all